# Traditional Machine Learning: Financial News Text Classification

## Project Overview
This notebook implements a comprehensive text classification system using traditional machine learning approaches for sentiment analysis of financial news. The project demonstrates the complete pipeline from data acquisition through model evaluation with professional-grade reporting.

## Key Components
- **Feature Extraction**: TF-IDF (Term Frequency-Inverse Document Frequency) vectorization
- **Classification Algorithms**: Naive Bayes and Logistic Regression
- **Evaluation Metrics**: Accuracy, Precision, Recall, F1-Score, and Confusion Matrices
- **Visualization**: Interactive Plotly-based performance dashboards and confusion matrices

## Dataset Information
- **Source**: Twitter Financial News Sentiment Dataset (Hugging Face)
- **Size**: ~4,800 training + ~1,200 validation samples
- **Classes**: 3 sentiment labels (Positive, Negative, Neutral)
- **Domain**: Financial news and market commentary

## Methodology
This project follows established ML best practices including data preprocessing, class balancing, train-test splitting, model evaluation, and comprehensive performance reporting.

## 1. Environment Setup and Dependencies

In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import re
import time

# Feature extraction and vectorization
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# Classification algorithms
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

# Evaluation metrics
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

print("✓ All dependencies imported successfully")

## 2. Data Loading and Initial Exploration

In [ ]:
# Load datasets from Hugging Face
base_url = 'https://huggingface.co/datasets/zeroshot/twitter-financial-news-sentiment/raw/main/'

train_df = pd.read_csv(base_url + 'sent_train.csv')
val_df = pd.read_csv(base_url + 'sent_valid.csv')

print(f'✓ Train: {len(train_df):,} samples')
print(f'✓ Val: {len(val_df):,} samples')
print(f'✓ Categories: {sorted(train_df["label"].unique().tolist())}')
print("\n" + "="*50)
print("Dataset Overview")
print("="*50)
print(train_df.head())
print(f"\nDataset shape: {train_df.shape}")
print(f"\nClass distribution:\n{train_df['label'].value_counts()}")

## 3. Text Preprocessing Pipeline

### Preprocessing Workflow:
The preprocessing pipeline applies the following transformations sequentially:
1. **URL Removal**: Eliminates hyperlinks and web addresses
2. **Stock Ticker Extraction**: Captures company symbols (e.g., $AAPL, $TSLA)
3. **Case Normalization**: Converts all text to lowercase
4. **Special Character Removal**: Removes non-alphabetic characters
5. **Whitespace Normalization**: Removes extra spaces
6. **Stop Word Removal**: Filters out common words with low semantic value

In [ ]:
def remove_urls(text):
    """Remove URLs and web addresses from text"""
    return re.sub(r'https?://\S+|www\.\S+', '', text)

def handle_company(text):
    """Temporarily replace company ticker symbols with placeholder"""
    return re.sub(r'\$\w+', '<code>', text)

def clean_text(text):
    """Remove special characters and extra whitespace"""
    text = re.sub(r'<code>', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def take_company(text):
    """Extract and list all company ticker symbols"""
    return re.findall(r'\$[A-Za-z]+', str(text))

# Comprehensive English stop words
STOP_WORDS = set([
    'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're",
    "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he',
    'him', 'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's",
    'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which',
    'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are',
    'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do',
    'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because',
    'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 
    'between', 'into', 'through', 'during', 'before', 'after', 
    'to', 'from', 'in', 'out', 'on', 'off', 'again',
    'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all',
    'each', 'few', 'more', 'most', 'other', 'some', 'such', 
    'only', 'own', 'same', 'so', 'than', 'too', 'very', 'can', 'will', 'just',
    'should', 'now', 'one', 'also'
])

print(f"✓ Preprocessing functions defined")
print(f"✓ Stop words vocabulary: {len(STOP_WORDS)} words")

In [ ]:
def preprocess_data(input_df):
    """
    Apply comprehensive text preprocessing to dataset
    
    Steps:
    1. Remove URLs
    2. Extract company tickers
    3. Lowercase conversion
    4. Clean special characters
    5. Remove duplicates
    6. Handle missing values
    
    Returns cleaned and processed dataframe
    """
    df = input_df.copy()
    
    # Apply preprocessing steps sequentially
    df['text'] = df['text'].apply(remove_urls)
    df['tickers'] = df['text'].apply(take_company)
    df['text'] = df['text'].apply(handle_company)
    df['text'] = df['text'].apply(lambda x: x.lower())
    df['text'] = df['text'].apply(clean_text)
    df['text'] = df['text'].apply(
        lambda x: ' '.join([word for word in x.split() if word not in STOP_WORDS])
    )
    
    # Remove duplicate records
    duplicate_count = df.astype(str).duplicated().sum()
    duplicate_pct = (duplicate_count / len(df)) * 100
    if duplicate_count > 0: 
        print(f"⚠️ Detected {duplicate_count} duplicates ({duplicate_pct:.2f}% of dataset)")
        df = df[~df.astype(str).duplicated()].reset_index(drop=True)
        print(f"✅ Duplicates removed. Remaining samples: {len(df)}")
    else:
        print("✅ No duplicates found in dataset\n")
    
    # Handle missing values
    missing_counts = df.isnull().sum()
    missing_pct = (df.isnull().sum() / len(df) * 100)

    missing_df = pd.DataFrame({
        'column': missing_counts[missing_counts > 0].index,
        'count': missing_counts[missing_counts > 0].values,
        'percentage': missing_pct[missing_counts > 0].values
    }).sort_values('percentage', ascending=False)
    
    if len(missing_df) > 0:
        print("Missing Values Summary:")
        print(missing_df)

        fig = go.Figure(data=[go.Bar(
            x=missing_df['column'],
            y=missing_df['percentage'],
            marker_color='#f093fb',
            text=[f"{p:.2f}%" for p in missing_df['percentage']],
            textposition='outside'
        )])

        fig.update_layout(
            title='Missing Values Distribution',
            xaxis_title='Features',
            yaxis_title='Missing Percentage (%)',
            width=800,
            height=400
        )
        print("\nRemoving rows with missing values...")
        df.dropna(inplace=True)
        df.reset_index(drop=True, inplace=True)
        print(f"✅ Dataset cleaned. Remaining samples: {len(df)}")
        fig.show()
    else:
        print("✅ No missing values detected\n")
    
    return df

print("✓ Preprocessing function defined")

In [ ]:
# Apply preprocessing to all datasets
print("\n--- Preprocessing Training Data ---")
train_df, test_df = train_test_split(
    train_df, 
    test_size=0.2, 
    random_state=42, 
    stratify=train_df['label']
)

print(f"Train Set: {len(train_df)} samples")
print(f"Test Set: {len(test_df)} samples")

print("\n--- Processing Train Set ---")
train_df = preprocess_data(train_df)
print("\n--- Processing Test Set ---")
test_df = preprocess_data(test_df)
print("\n--- Processing Validation Set ---")
val_df = preprocess_data(val_df)

## 4. Feature Extraction with TF-IDF

### TF-IDF (Term Frequency-Inverse Document Frequency)
TF-IDF is a numerical statistic that reflects how important a word is to a document in a collection of documents. It balances:
- **Term Frequency (TF)**: How often a term appears in a document
- **Inverse Document Frequency (IDF)**: How rare a term is across all documents

### Configuration Parameters:
- `max_features`: Maximum number of features (vocabulary size)
- `ngram_range`: Combinations of consecutive terms (unigrams + bigrams)
- `min_df`: Minimum document frequency to include a term
- `max_df`: Maximum document frequency to filter out common terms

In [ ]:
# Initialize and fit TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),  # Unigrams (single words) + Bigrams (word pairs)
    min_df=2,            # Minimum 2 documents must contain the term
    max_df=0.8           # Maximum 80% of documents can contain the term
)

print("\n" + "="*70)
print("FEATURE EXTRACTION: TF-IDF VECTORIZATION")
print("="*70 + "\n")

# Fit on training data and transform all datasets
start = time.time()
X_train = vectorizer.fit_transform(train_df['text'])
X_test = vectorizer.transform(test_df['text'])
X_val = vectorizer.transform(val_df['text'])
elapsed = time.time() - start

# Extract feature names
feature_names = vectorizer.get_feature_names_out()

print(f'✓ Vocabulary Size: {len(feature_names):,} features')
print(f'✓ Training Data Shape: {X_train.shape}')
print(f'✓ Test Data Shape: {X_test.shape}')
print(f'✓ Validation Data Shape: {X_val.shape}')
print(f'✓ Extraction Time: {elapsed:.2f}s')
print(f'\n✓ Sample Features (first 20): {list(feature_names[:20])}')

## 5. Model Definition and Training Functions

In [ ]:
def train_classifier(name, model, X_train, y_train, X_test, y_test):
    """
    Train a classifier and evaluate its performance
    
    Parameters:
    -----------
    name : str
        Name of the classifier for display
    model : sklearn classifier
        Initialized classifier instance
    X_train, y_train : Training data and labels
    X_test, y_test : Test data and labels
    
    Returns:
    --------
    dict : Dictionary containing model and performance metrics
    """
    
    print("="*70)
    print(f"{name.upper()}")
    print("="*70)
    
    # Training phase
    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start
    
    # Prediction phase
    start = time.time()
    y_pred = model.predict(X_test)
    inference_time = time.time() - start
    
    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average='weighted', zero_division=0
    )
    cm = confusion_matrix(y_test, y_pred)
    
    # Display results
    print(f"\n⏱️  Training Time: {train_time:.4f}s")
    print(f"⏱️  Inference Time: {inference_time:.4f}s")
    print(f"\n📊 Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"📊 Precision: {precision:.4f} ({precision*100:.2f}%)")
    print(f"📊 Recall:    {recall:.4f} ({recall*100:.2f}%)")
    print(f"📊 F1-Score:  {f1:.4f} ({f1*100:.2f}%)")
    print()
    
    # Store results
    return {
        'name': name,
        'model': model,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'train_time': train_time,
        'inference_time': inference_time,
        'inference_speed': len(y_test) / inference_time if inference_time > 0 else 0,
        'y_pred': y_pred,
        'confusion_matrix': cm
    }

print("✓ Training function defined")

## 6. Train Classification Models

In [ ]:
print("\n" + "="*70)
print("🚀 TEXT CLASSIFICATION WITH TRADITIONAL ML METHODS")
print("="*70 + "\n")

# Get target labels
y_train = train_df['label']
y_test = test_df['label']
y_val = val_df['label']

# Initialize classifiers
classifiers = [
    ('Naive Bayes', MultinomialNB()),
    ('Logistic Regression', LogisticRegression(max_iter=1000, random_state=42))
]

# Train and evaluate each classifier
results = []
for name, model in classifiers:
    result = train_classifier(name, model, X_train, y_train, X_val, y_val)
    results.append(result)
    print()

## 7. Visualization Functions for Performance Analysis

In [ ]:
def plot_confusion_matrix_plotly(cm, labels, title):
    """
    Create interactive Plotly confusion matrix heatmap
    
    Parameters:
    -----------
    cm : np.ndarray
        Confusion matrix from sklearn
    labels : list
        Class labels
    title : str
        Title for the heatmap
    
    Returns:
    --------
    plotly.graph_objects.Figure : Interactive heatmap
    """
    # Convert to native Python types for Plotly compatibility
    if hasattr(cm, 'tolist'):
        cm = cm.tolist()
    
    # Convert labels to strings
    labels_str = [str(L) for L in labels]
    
    fig = go.Figure(data=go.Heatmap(
        z=cm,
        x=labels_str,
        y=labels_str,
        colorscale='Blues',
        showscale=True,
        text=cm,
        texttemplate='%{text}',
        textfont={"size": 14},
        hovertemplate='True: %{y}<br>Predicted: %{x}<br>Count: %{z}<extra></extra>'
    ))
    
    fig.update_layout(
        title=f'Confusion Matrix - {title}',
        xaxis_title='Predicted Label',
        yaxis_title='True Label',
        yaxis_autorange='reversed',
        width=600,
        height=500,
        font=dict(size=12)
    )
    
    return fig


def create_comparison_chart(results):
    """
    Create side-by-side comparison chart for classifiers
    
    Returns:
    --------
    plotly.graph_objects.Figure : Comparison visualization
    """
    names = [r['name'] for r in results]
    accuracies = [r['accuracy']*100 for r in results]
    train_times = [r['train_time'] for r in results]
    
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Accuracy Comparison', 'Training Time Comparison'),
        specs=[[{"type": "bar"}, {"type": "bar"}]]
    )
    
    # Accuracy bars
    fig.add_trace(
        go.Bar(x=names, y=accuracies, name='Accuracy (%)',
               marker_color='rgb(102, 126, 234)',
               text=[f'{a:.2f}%' for a in accuracies],
               textposition='outside'),
        row=1, col=1
    )
    
    # Training time bars
    fig.add_trace(
        go.Bar(x=names, y=train_times, name='Time (s)',
               marker_color='rgb(245, 135, 108)',
               text=[f'{t:.4f}s' for t in train_times],
               textposition='outside'),
        row=1, col=2
    )
    
    fig.update_layout(
        title_text="Classifier Performance Comparison",
        showlegend=False,
        height=400,
        font=dict(size=12)
    )
    
    fig.update_yaxes(title_text="Accuracy (%)", row=1, col=1)
    fig.update_yaxes(title_text="Time (seconds)", row=1, col=2)
    
    return fig

print("✓ Visualization functions defined")

## 8. Generate Visualizations

In [ ]:
# Performance comparison chart
comparison_fig = create_comparison_chart(results)
comparison_fig.show()

In [ ]:
# Confusion matrices for each classifier
print("\n" + "="*70)
print("CONFUSION MATRICES")
print("="*70 + "\n")

labels = sorted(y_val.unique())

for result in results:
    cm_fig = plot_confusion_matrix_plotly(result['confusion_matrix'], labels, result['name'])
    cm_fig.show()
    print()

## 9. Comprehensive HTML Report Generation

In [ ]:
def generate_html_report(results, y_test):
    """
    Generate professional HTML report with interactive visualizations
    
    Parameters:
    -----------
    results : list
        List of result dictionaries from model training
    y_test : pd.Series
        Test labels for confusion matrix generation
    
    Returns:
    --------
    str : HTML report content
    """
    labels = sorted(y_test.unique())
    
    # CSS styling
    css_style = """
        body { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 0; padding: 0; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); }
        .container { max-width: 1400px; margin: 20px auto; background: white; padding: 40px; border-radius: 12px; box-shadow: 0 10px 40px rgba(0,0,0,0.2); }
        h1 { color: #1a1a1a; text-align: center; margin-bottom: 5px; font-size: 2.5em; }
        .subtitle { text-align: center; color: #666; margin-bottom: 30px; font-size: 1.1em; }
        h2 { color: #667eea; margin-top: 40px; font-size: 1.8em; border-bottom: 3px solid #667eea; padding-bottom: 10px; }
        .summary { background: linear-gradient(135deg, #f5f7fa 0%, #c3cfe2 100%); padding: 30px; border-radius: 10px; margin: 30px 0; }
        .summary table { width: 100%; border-collapse: collapse; font-size: 14px; }
        .summary th { background: #667eea; color: white; padding: 15px; text-align: center; font-weight: 700; border-radius: 5px 5px 0 0; }
        .summary td { padding: 12px; text-align: center; border-bottom: 2px solid #e9ecef; }
        .summary td:first-child { text-align: left; font-weight: 600; }
        .summary tr:hover { background: rgba(102, 126, 234, 0.1); }
        .best { background: #d4edda !important; font-weight: bold; color: #155724; border-radius: 4px; }
        .chart-container { margin: 40px 0; background: #f8f9fa; padding: 30px; border-radius: 10px; border-left: 5px solid #667eea; }
        .cm-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(600px, 1fr)); gap: 30px; margin-top: 30px; }
        .metric-box { background: white; padding: 20px; border-radius: 8px; border-top: 4px solid #667eea; margin: 10px 0; }
        .metric-label { color: #666; font-size: 0.9em; font-weight: 600; }
        .metric-value { color: #667eea; font-size: 2em; font-weight: bold; }
    """
    
    html_parts = [f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Financial News Text Classification Report</title>
    <script src="https://cdn.plot.ly/plotly-2.26.0.min.js"></script>
    <style>
        {css_style}
    </style>
</head>
<body>
    <div class="container">
        <h1>📊 Financial News Text Classification</h1>
        <p class="subtitle">Traditional Machine Learning Models (TF-IDF + Naive Bayes/Logistic Regression)</p>
        
        <div class="summary">
            <h2>Performance Metrics Comparison</h2>
            <table>
                <tr>
                    <th>Model</th>
                    <th>Accuracy</th>
                    <th>Precision</th>
                    <th>Recall</th>
                    <th>F1-Score</th>
                    <th>Training Time</th>
                    <th>Inference Speed</th>
                </tr>
"""]
    
    # Find best values for highlighting
    best_acc = max(r['accuracy'] for r in results)
    best_prec = max(r['precision'] for r in results)
    best_rec = max(r['recall'] for r in results)
    best_f1 = max(r['f1_score'] for r in results)
    best_train = min(r['train_time'] for r in results)
    best_speed = max(r['inference_speed'] for r in results)
    
    # Add result rows
    for r in results:
        acc_class = ' class="best"' if r['accuracy'] == best_acc else ''
        prec_class = ' class="best"' if r['precision'] == best_prec else ''
        rec_class = ' class="best"' if r['recall'] == best_rec else ''
        f1_class = ' class="best"' if r['f1_score'] == best_f1 else ''
        train_class = ' class="best"' if r['train_time'] == best_train else ''
        speed_class = ' class="best"' if r['inference_speed'] == best_speed else ''
        
        html_parts.append(f"""
                <tr>
                    <td><strong>{r['name']}</strong></td>
                    <td{acc_class}>{r['accuracy']*100:.2f}% ✓</td>
                    <td{prec_class}>{r['precision']*100:.2f}% ✓</td>
                    <td{rec_class}>{r['recall']*100:.2f}% ✓</td>
                    <td{f1_class}>{r['f1_score']*100:.2f}% ✓</td>
                    <td{train_class}>{r['train_time']:.4f}s</td>
                    <td{speed_class}>{r['inference_speed']:.2f} samples/s</td>
                </tr>
""")
    
    html_parts.append("""
            </table>
        </div>
        
        <div class="chart-container">
            <h2>Performance Comparison Charts</h2>
            <div id="comparison-chart"></div>
        </div>
        
        <div class="chart-container">
            <h2>Confusion Matrices</h2>
            <div class="cm-grid">
""")
    
    # Add placeholders for confusion matrices
    for i, r in enumerate(results):
        html_parts.append(f'                <div id="cm-{i}"></div>\n')
    
    html_parts.append("""
            </div>
        </div>
        
        <div style="margin-top: 50px; padding: 20px; background: #f8f9fa; border-radius: 8px; text-align: center; color: #666;">
            <p><strong>Report Generated:</strong> Traditional Machine Learning Classification Pipeline</p>
            <p>Dataset: Twitter Financial News Sentiment | Models: Naive Bayes, Logistic Regression</p>
        </div>
    </div>
    
    <script>
""")
    
    # Add comparison chart
    comp_fig = create_comparison_chart(results)
    html_parts.append(f"        var compData = {comp_fig.to_json()};\n")
    html_parts.append("        Plotly.newPlot('comparison-chart', compData.data, compData.layout);\n\n")
    
    # Add confusion matrices
    for i, r in enumerate(results):
        cm_fig = plot_confusion_matrix_plotly(r['confusion_matrix'], labels, r['name'])
        html_parts.append(f"        var cmData{i} = {cm_fig.to_json()};\n")
        html_parts.append(f"        Plotly.newPlot('cm-{i}', cmData{i}.data, cmData{i}.layout);\n\n")
    
    html_parts.append("""
    </script>
</body>
</html>
""")
    
    return ''.join(html_parts)

print("✓ HTML report generation function defined")

In [ ]:
# Generate and save HTML report
print("\n" + "="*70)
print("📄 GENERATING HTML REPORT")
print("="*70 + "\n")

html_report = generate_html_report(results, y_val)

with open('classification_results.html', 'w', encoding='utf-8') as f:
    f.write(html_report)

print("✓ Report saved to: classification_results.html")
print("✓ File location: ", end="")
import os
print(os.path.abspath('classification_results.html'))

## 10. Summary and Conclusions

In [ ]:
print("\n" + "="*70)
print("✅ CLASSIFICATION PIPELINE COMPLETE")
print("="*70 + "\n")

# Summary statistics
print("📋 EXECUTION SUMMARY")
print("-" * 70)
print(f"Total Models Trained: {len(results)}")
print(f"Training Dataset Size: {len(train_df):,} samples")
print(f"Test Dataset Size: {len(y_val):,} samples")
print(f"Feature Dimension: {X_train.shape[1]:,} features")
print(f"Number of Classes: {len(labels)}")
print(f"\nBest Model: {results[0]['name']} (Accuracy: {max(r['accuracy'] for r in results)*100:.2f}%)")
print(f"\n📁 Output Files Generated:")
print(f"   ✓ classification_results.html (Interactive Report)")
print("\n" + "="*70)